
# Processing Silver layer for Account_move model

Process for create a silver layer for the account_move model, for build that I created a temp view, this temp view has some features like compute the following fields:
- company_id: array
- journal_id: array
- partner_id: String, extract the information from the field, id and name
- cast all date fields
- Quility row_over() for get only the last one record created.

and get the merge code by account_move_id_ for avoid the duplicate information.

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_account_move AS
SELECT 
  id AS account_move_id,
  company_id[1] AS company_name, 
  journal_id[0] AS journal_id,
  journal_id[1] AS journal_name,
  CAST(
        CASE 
            WHEN partner_id IS NULL OR partner_id = 'false' THEN NULL
            ELSE get_json_object(partner_id, '$[0]')
        END AS INT
    ) AS customer_id,
    COALESCE(
        CASE 
            WHEN partner_id = 'false' THEN NULL
            ELSE get_json_object(partner_id, '$[1]')
        END, 
        'Cliente General'
    ) AS customer_name,
  name AS move_name,
  state AS move_state,
  ref AS reference,
  move_type,
  payment_state,
  CAST(date AS DATE) AS account_date,
  CAST(create_date AS TIMESTAMP) AS created_at,
  write_uid[0] AS updated_by, 
  CAST(write_date AS TIMESTAMP) AS updated_at,
  processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.account_move
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id 
    ORDER BY write_date DESC, processing_timestamp DESC
) = 1

### Is fundamental to create the table with the structure that coincidence with 100% with the view, because generate an error multiple times.


In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.account_move (
  account_move_id INT,
  company_name STRING,
  journal_id INT,
  journal_name STRING,
  customer_id INT,
  customer_name STRING,
  move_name STRING,
  reference STRING,
  move_type STRING,
  move_state STRING,
  payment_state STRING,
  account_date DATE,
  created_at TIMESTAMP,
  updated_at TIMESTAMP,
  updated_by INT,
  state STRING,
  bronze_ingested_at TIMESTAMP,
  silver_processed_at TIMESTAMP
) USING DELTA;



MERGE INTO bakery_pos_develop.silver.account_move AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_account_move
) AS source
ON target.account_move_id = source.account_move_id
WHEN MATCHED THEN
  UPDATE SET
    company_name = source.company_name,
    journal_id = source.journal_id,
    journal_name = source.journal_name,
    customer_id = source.customer_id,
    customer_name = source.customer_name,
    move_name = source.move_name,
    reference = source.reference,
    move_type = source.move_type,
    move_state = source.move_state,
    payment_state = source.payment_state,
    account_date = source.account_date,
    created_at = source.created_at,
    updated_at = source.updated_at,
    updated_by = source.updated_by,
    state = 'updated',
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN 
  INSERT (
    account_move_id,
    company_name,
    journal_id,
    journal_name,
    customer_id,
    customer_name,
    move_name,
    reference,
    move_type,
    move_state,
    payment_state,
    account_date,
    created_at,
    updated_at,
    updated_by,
    state,
    bronze_ingested_at,
    silver_processed_at
  )
  VALUES (
    source.account_move_id,
    source.company_name,
    source.journal_id,
    source.journal_name,
    source.customer_id,
    source.customer_name,
    source.move_name,
    source.reference,
    source.move_type,
    source.move_state,
    source.payment_state,
    source.account_date,
    source.created_at,
    source.updated_at,
    source.updated_by,
    'inserted',
    source.bronze_ingested_at,
    source.silver_processed_at
  );

In [0]:
import datetime
import pytz

bogota_tz = pytz.timezone("America/Bogota")

year = datetime.datetime.now(tz=bogota_tz).strftime("%Y")
day = datetime.datetime.now(tz=bogota_tz).strftime("%d")
month = datetime.datetime.now(tz=bogota_tz).strftime("%m")

In [0]:
%sql

CREATE TEMP VIEW stg_account_move_deletes AS
SELECT *, current_timestamp() AS silver_processed_at FROM read_files(
  's3://bakery-pos-bk-978430483777-us-east-2-an/odoo_deletes/${year}/${month}/${day}',
  format => 'json',
  multiLine => 'true'
)

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.account_move AS target
USING stg_account_move_deletes AS source
ON target.account_move_id = source.record_id
WHEN MATCHED THEN
  UPDATE SET
    updated_by = source.user_id,
    state = 'deleted',
    silver_processed_at = source.silver_processed_at;